# 04.4 · q-transfer — cross-task multi-predicate transplant

**Question.** Does the goal heads' *query* at the pivot carry "what to do next", and in what form? Transplant the SRC
pivot query into DST and see which task DST writes: one predicted by **set progress** (the query encodes *which* tasks
are done) or by **count** (it encodes *how many* are done).

**Notation.** Capital letters are tasks (predicates, knob `TASKS`); `'` = already written in the prefill; `→` = the
next task. Code variables are lettered **sequentially** (`a, b, c, …` in the order lines are written; the stub takes
the next letter) — unrelated to the task letters and never the task's list position.

**Configurations** (`USE_KV`):

| | prompts | DST writes (clean) | q-transfer predicts |
|---|---|---|---|
| `USE_KV=False` | `p_src A'CB' → C` · `p_dst A'BD → B` | B | set progress **D**, count **D** |
| `USE_KV=True` (04.3) | `p_src A'CB' → C` · `p_dst A'XY → X` · `p_kv A'BD → B` | X | kv-only **B** · kv+q: set **D**, count **Y** |

*Set progress* = first task of the list DST's goal heads read (p_kv's with kv, else DST's own) written in neither DST
nor SRC. *Count* = DST's slot `|SRC_DONE|` on DST's own listing. SRC's own target C is in neither p_dst nor p_kv.

**Pivot** (`PIVOT`): `"indent"` — the prefill ends at the trailing indent; `"var_eq"` — it ends at the next variable's
` =` (e.g. `    b =`), giving three anchors (indent, var, eq) on each side.

**Interventions** (goal heads; `N_GOAL_HEADS` = 122 or 250):
- **q-transfer** — SRC's pre-RoPE query at each of `PREFILL_ANCHORS` → the same anchor in DST (prefill), then one query
  (`DECODE_Q`) re-clamped for the first `N_STEER − 1` decode steps.
- **kv-transfer** (`USE_KV`) — p_kv's task-list key+value on `KV_HEADS` → DST's task-list span (GQA-corrected
  head-wise routing, `pad` alignment: newline tokens spliced before the shorter span, so both spans have the same
  length and every token moves).
- **Sufficiency ablation** (`ATTN_ABL_SUFFICIENCY`, `ATTN_ABL_LAYER` = l) — on every patched run, all heads in
  layers > l except the goal heads are blinded from the task span through the pivot (they keep running), as in the
  paper's `blind_span` sufficiency test.
- **Attention restriction** (`PATCH_ATTN_RESTRICT`) — on every patched run (kv-only and kv+q with kv, else the
  q-patched run, incl. their read-outs, gaze and the closed loop's patched steps) the PATCH_HEADS can read only the
  task span (the p_kv window with kv, else p_dst's list) + position 0; generated tokens stay visible.
- **OV boost** (`BOOST_ALPHA`, both modes) — scales the patched heads' OV write at the current token on every
  patched forward (PATCH_HEADS, ∪ KV_HEADS with kv); 2.0 = the batch script's `--alphas 2`. The clean destination always runs on the unpadded DST.
- **closed loop** — instead of re-clamping one fixed query, regenerate the goal heads' query at every step from the
  goal encoding (see the closed-loop section).

**Sections:** prompts → runs (generation + GoalScope read-out) → causality read-out → gaze → closed loop
(+ its read-outs and per-step gaze).

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

CUDA = "1"                                   # free GPU(s); Coder-32B fits on one 80 GB GPU
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")                     # repo root, so `src` is importable
import json, torch, numpy as np
import src.diff.functional as FUNC
from src.diff.functional import *            # helpers into scope
import src.diff.data as DATA
from src.attention import get_attention_matrices
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from rich.markup import escape
from src.utils.experiment_utils import set_seed

FUNC.setup("Qwen/Qwen2.5-Coder-32B-Instruct")   # loads the model + the 122 goal heads (lam20 · thr 0.9 · 25-56)
mt = FUNC.mt


def pivot_of(prompt, neg_shift=-1):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] + neg_shift


def show(prompt, title, emph_span=None, piv=-1):
    kw = dict(patch_pos=[piv])
    if emph_span is not None:
        kw["emphasize_tokens"] = range(*emph_span)
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, **kw)), title=f"[bold]{title}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))

2026-10-07 12:19:38 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Knobs

In [2]:
# ── knobs ────────────────────────────────────────────────────────────────────
N_GOAL_HEADS = 250        # 122: lam20 · thr 0.9 · layers 25-56 (paper set) | 250: lam20 top-250 (best kv transfer)
DATATYPE     = "list[str]"
NAMING       = "letters"  # result variable names in the code: "letters" (a, b, c) | "natural" (descriptive)

# TASKS = {                
#     "A": "count_long",
#     "B": "shortest_word_length",
#     "C": "distinct_count",
#     "D": "count_starts_vowel",
#     "X": "total_length",       
#     "Y": "has_palindrome",
# }

# TASKS = {
#     "A": "distinct_count",
#     "B": "has_palindrome",
#     "C": "total_length",
#     "D": "count_starts_vowel",
#     "X": "total_length",   
#     "Y": "any_empty",    
# }

TASKS = {
    "A": "longest_word_length",
    "B": "distinct_count",
    "C": "count_capitalized",
    "D": "total_vowels",
    "X": "total_length",
    "Y": "count_ends_consonant",
}

# Design (04.3 notation, ' = written):
#   USE_KV=False:  p_src a'cb' -> c   p_dst a'bd -> b                      q-transfer: set progress -> d, count -> d
#   USE_KV=True :  p_src a'cb' -> c   p_dst a'xy -> x   p_kv a'bd -> b     kv-only -> b · kv+q: set -> d, count -> y
USE_KV   = True          # True: also transplant KV's task-list k+v onto DST's task list (04.3 setup)
SRC_ORDER, SRC_DONE = "ACB", "AB"   # SRC: listed tasks (in order), written tasks -> target = first listed not written
DST_ORDER, DST_DONE = ("AXY", "A") if USE_KV else ("ABD", "A")   # DST: listed tasks, written tasks -> frontier = first not written

PIVOT    = "var_eq"       # "indent": prefill ends at the trailing indent · "var_eq": ends at the next var's " =" (`c =`)
PREFILL_ANCHORS = ("indent", "var", "eq")   # prefill: SRC query at each anchor -> the SAME anchor in DST ("var"/"eq" need PIVOT="var_eq")
DECODE_Q = "eq"          # query re-clamped at decode steps: "eq" (SRC " =" query) | "mean" (avg of SRC indent/var/eq queries)

# PIVOT = "indent"
# PREFILL_ANCHORS = ("indent",)
# DECODE_Q = "indent"

N_STEER  = 32             # forward passes that carry the SRC query: 1 = DST pivot only (prefill); k = pivot + first k-1 decode steps
MAX_NEW  = 96             # generation budget per run
KV_ORDER = "ABD"          # KV's task list; its written tasks = DST_DONE, so only the task list differs from DST
KV_HEADS = None           # heads whose k+v are transplanted (None = GOAL_HEADS)
ATTN_ABL_SUFFICIENCY = False  # True: sufficiency ablation on every patched run (kv-only + kv+q with kv, else the
                              # q-patched run) — every head in layers > ATTN_ABL_LAYER EXCEPT the goal heads is BLINDED
                              # from the task span through the pivot (it keeps running; the paper's `blind_span`,
                              # scripts/evaluation/eval_ablation_sufficiency.py, which uses layers >= --from_layer)
ATTN_ABL_LAYER = 40           # l: ablate non-goal heads in layers > l
PATCH_ATTN_RESTRICT = False   # True: on every patched run (kv-only + kv+q with kv, else the q-patched run), the
                              # PATCH_HEADS attend only to the task span (p_kv's, inside DST, with kv; else p_dst's)
                              # plus position 0 — every other prompt key up to the pivot is masked out
BOOST_ALPHA = 2.0         # OV-write boost at the current token on EVERY patched forward (1.0 = off): PATCH_HEADS,
                          # ∪ KV_HEADS when USE_KV — the batch script's --alphas (paper q-transfer: 2.0)
KV_ALIGN = "pad"          # kv span alignment: "pad" | "right"
RO_KW    = dict(strategy="additive_norm", alpha=0.1, layers=(30, 41))   # GoalScope read-out config (paper default)

# ── derived ──────────────────────────────────────────────────────────────────
if N_GOAL_HEADS == 122:
    GOAL_HEADS = list(FUNC.GOAL_HEADS)
elif N_GOAL_HEADS == 250:
    GOAL_HEADS = [tuple(h) for h in json.load(open(
        "../../head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json"))]
else:
    raise ValueError("N_GOAL_HEADS must be 122 or 250")

READOUT_HEADS = GOAL_HEADS

#! overwrite goal heads
# GOAL_HEADS = [(l, h) for l in range(35, 55) for h in range(mt.n_heads)]

PATCH_HEADS = GAZE_HEADS = GOAL_HEADS 
KV_HEADS = KV_HEADS or GOAL_HEADS

bank = DATA.QTASKS[DATATYPE]
by = {p.id: p for p in bank.predicates}
assert all(TASKS[L] in by for L in SRC_ORDER + DST_ORDER), "unknown predicate id in TASKS"
assert set(SRC_DONE) <= set(SRC_ORDER) and set(DST_DONE) <= set(DST_ORDER)
TARGET   = next(L for L in SRC_ORDER if L not in SRC_DONE)    # SRC's next task
FRONTIER = next(L for L in DST_ORDER if L not in DST_DONE)    # DST's own next task
TN = lambda L: f"{L} = {TASKS[L]!r}" if L else "None"   # label: letter + task name
# q-transfer predictions. set progress: first task of the list DST's goal heads read (KV's with kv, else DST's own)
# done in neither DST nor SRC · count: DST slot |SRC_DONE| on DST's own listing
_PLIST = KV_ORDER if USE_KV else DST_ORDER
H1_NEXT = next((L for L in _PLIST if L not in set(DST_DONE) | set(SRC_DONE)), None)
COUNT_NEXT = DST_ORDER[len(SRC_DONE)] if len(SRC_DONE) < len(DST_ORDER) else None
if USE_KV:
    assert set(DST_DONE) <= set(KV_ORDER) and all(TASKS[L] in by for L in KV_ORDER), "KV_ORDER must contain DST_DONE"
    KV_NEXT = next(L for L in KV_ORDER if L not in DST_DONE)                         # kv-only expectation
    print(f"KV {{{','.join(KV_ORDER)}}} · kv-only expects {TN(KV_NEXT)} · kv+q: set progress -> {TN(H1_NEXT)}, "
          f"count -> {TN(COUNT_NEXT)} (SRC identity -> {TN(TARGET)})")
print(f"q-transfer predicts: set progress -> {TN(H1_NEXT)} · count -> {TN(COUNT_NEXT)}")
print(f"{len(GOAL_HEADS)} goal heads · SRC target {TN(TARGET)} · DST frontier {TN(FRONTIER)} · "
      f"target in DST: {TARGET in DST_ORDER} · N_STEER={N_STEER}")
for L in sorted(set(SRC_ORDER + DST_ORDER + (KV_ORDER if USE_KV else ""))):
    print(f"  {L} = {TASKS[L]:<22} {by[TASKS[L]].description}")

KV {A,B,D} · kv-only expects B = 'distinct_count' · kv+q: set progress -> D = 'total_vowels', count -> Y = 'count_ends_consonant' (SRC identity -> C = 'count_capitalized')
q-transfer predicts: set progress -> D = 'total_vowels' · count -> Y = 'count_ends_consonant'
250 goal heads · SRC target C = 'count_capitalized' · DST frontier X = 'total_length' · target in DST: False · N_STEER=32
  A = longest_word_length    the length of the longest word
  B = distinct_count         the number of distinct words
  C = count_capitalized      the number of capitalized words
  D = total_vowels           the total number of vowels across all words
  X = total_length           the total number of characters across all words
  Y = count_ends_consonant   the number of words that end with a consonant


## SRC and DST prompts

Prompts as the model sees them, with the pivot marked. With `USE_KV=True`, p_dst and p_kv are shown **aligned**:
task-list span = yellow, the exact tokens whose k+v move = orange, spliced `pad` newlines = `⏎pad` (grey),
pivot = salmon.

In [3]:
# ── SRC / DST prompts ────────────────────────────────────────────────────────
class SeqQTI(DATA.QTransferInstance):
    """`letters` naming is SEQUENTIAL (DATA._seq_letter): prefilled lines are a, b, c, ... in the order they are
    written and the frontier stub takes the next letter — never the task's list position. (The base
    QTransferInstance is still positional, which leaks list position whenever the prefill is not a list prefix.)"""
    def var_of(self, pred_id):
        if self.naming != "letters":
            return super().var_of(pred_id)
        return DATA._seq_letter(self.task.predicates, self.src_progress, pred_id)


def make_inst(order, done, target):
    task = DATA.dataclasses.replace(bank, id="xtask", predicates=tuple(by[TASKS[L]] for L in order))
    ids = [TASKS[L] for L in order if L in done]      # written in list order (matches _seq_letter)
    return SeqQTI(task=task, var_name=bank.var_names[0], presentation="listed",
                                  src_lang="python", dst_lang="python", src_progress=ids, dst_progress=ids,
                                  expected_next=TASKS[target], naming=NAMING, meta={"datatype": DATATYPE})


src_inst = make_inst(SRC_ORDER, SRC_DONE, TARGET)
dst_inst = make_inst(DST_ORDER, DST_DONE, FRONTIER)
def build_prompt(inst, parts, next_task):
    q, pf = parts
    if PIVOT == "var_eq":            # stub the next task's (sequential) variable up to " =" -> pivot = " ="
        pf = pf + f"{inst.var_of(TASKS[next_task])} ="
    elif PIVOT != "indent":
        raise ValueError(f"PIVOT must be 'indent' or 'var_eq', got {PIVOT!r}")
    return get_prompt(q, DATA._finalize_prefill(pf))


src_prompt = build_prompt(src_inst, src_inst.src_parts(), TARGET)
dst_prompt = build_prompt(dst_inst, dst_inst.dst_parts(), FRONTIER)
piv_src, piv_dst = pivot_of(src_prompt), pivot_of(dst_prompt)
src_tok = prepare_input(prompts=src_prompt, tokenizer=mt)
dst_tok = prepare_input(prompts=dst_prompt, tokenizer=mt)

tasks_str = lambda order: "{" + ",".join(order) + "}"
print(f"pivot token: SRC {mt.tokenizer.decode(src_tok['input_ids'][0][piv_src:piv_src+1])!r} · "
      f"DST {mt.tokenizer.decode(dst_tok['input_ids'][0][piv_dst:piv_dst+1])!r}")
show(src_prompt, f"SRC · {tasks_str(SRC_ORDER)} · {','.join(SRC_DONE)} done · target {TN(TARGET)} · pivot@{piv_src}")
if not USE_KV:                                       # with kv, DST is shown aligned (with spans) below
    show(dst_prompt, f"DST · {tasks_str(DST_ORDER)} · {','.join(DST_DONE)} done · frontier {TN(FRONTIER)} · pivot@{piv_dst}")

# ── optional KV prompt (04.3): its task-list k+v -> DST's task list ───────────
dst_tok_clean, piv_dst_clean = dst_tok, piv_dst      # clean DST, never padded
task_text = lambda order: "\n".join(f"({i + 1}) {by[TASKS[L]].description}" for i, L in enumerate(order))


def item_positions(prompt, order):
    return {L: list(range(*token_span(prompt, by[TASKS[L]].description))) for L in order}


PATCHED_ITEMS = item_positions(dst_prompt, DST_ORDER)   # {letter: DST positions} the patched runs' gaze is read over
_FROM_DST = {}                                           # DST position -> p_kv position, inside the kv window
PATCHED_LABEL = "DST tasks"


def show_spans(tok, title, task_span=None, kv_span=None, pad=(), piv=-1):
    """Render a tokenized prompt exactly as the model sees it (from 04.3): task_span -> bright_yellow,
    kv_span -> black on dark_orange (the exact tokens whose k+v move), pad -> the spliced "\\n" alignment tokens
    (black on grey62, shown as ⏎pad), pivot -> salmon. Priority pivot > pad > kv > task."""
    ids = tok["input_ids"][0].tolist()
    n = len(ids); pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set()
    kv = set(range(*kv_span)) if kv_span else set()
    pad = set(pad)
    # sentinel markers, escape the joined string once, then swap in rich tags (per-token escaping drops brackets)
    PO, PC, DO, DC, KO, KC, TO, TC = "\x00<", "\x00>", "\x03<", "\x03>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos:
            s = PO + s + PC
        elif i in pad:
            s = DO + "⏎pad" + DC + "\n"
        elif i in kv:
            s = KO + s + KC
        elif i in task:
            s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    for a, b, tag in ((PO, PC, "bold black on salmon1"), (DO, DC, "black on grey62"),
                      (KO, KC, "bold black on dark_orange"), (TO, TC, "bright_yellow")):
        txt = txt.replace(a, f"[{tag}]").replace(b, f"[/{tag}]")
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))


def patched_tok_text(j):
    # token at DST position j as the patched runs' goal heads see it: inside the kv window that is p_kv's token
    # (its k+v were transplanted there), elsewhere DST's own
    if j in _FROM_DST:
        return mt.tokenizer.decode([int(kv_tok["input_ids"][0][_FROM_DST[j]])])
    return mt.tokenizer.decode([int(dst_tok["input_ids"][0][j])])


if USE_KV:
    kv_inst = make_inst(KV_ORDER, DST_DONE, KV_NEXT)
    kv_prompt = build_prompt(kv_inst, kv_inst.dst_parts(), KV_NEXT)
    kv_task, dst_task = task_text(KV_ORDER), task_text(DST_ORDER)
    KV0, DST0 = token_span(kv_prompt, kv_task), token_span(dst_prompt, dst_task)
    kv_tok, dst_tok, KV_POS, DST_POS, KV_SPAN, DST_SPAN = align_kv_spans(
        mt, kv_prompt, dst_prompt, KV0, DST0, mode=KV_ALIGN, src_text=kv_task, dst_text=dst_task)
    piv_dst = dst_tok["input_ids"].shape[1] - 1          # patched runs use the (possibly padded) aligned DST
    _kvh = heads_per_layer(KV_HEADS, "k", layers=set(FUNC._layers_of(KV_HEADS)))
    KV_K = cache_src_proj(kv_tok, KV_POS, sorted(_kvh), "k")
    KV_V = cache_src_proj(kv_tok, KV_POS, sorted(_kvh), "v")
    # KV items -> DST positions through the aligned window (pad splices newlines right before the shorter span)
    _kv_pad = (KV_SPAN[1] - KV_SPAN[0]) - (KV0[1] - KV0[0])
    _to_dst = dict(zip(KV_POS, DST_POS))
    _FROM_DST = dict(zip(DST_POS, KV_POS))
    PATCHED_ITEMS = {L: [_to_dst[p + _kv_pad] for p in ps if p + _kv_pad in _to_dst]
                     for L, ps in item_positions(kv_prompt, KV_ORDER).items()}
    PATCHED_LABEL = "p_kv tasks"
    # alignment pads: "pad" splices newline ids right BEFORE the shorter span -> the first n positions of that span
    _dst_pad = (DST_SPAN[1] - DST_SPAN[0]) - (DST0[1] - DST0[0])
    KV_PADS = list(range(KV_SPAN[0], KV_SPAN[0] + _kv_pad)) if KV_ALIGN == "pad" else []
    DST_PADS = list(range(DST_SPAN[0], DST_SPAN[0] + _dst_pad)) if KV_ALIGN == "pad" else []
    print(f"align={KV_ALIGN}: {len(KV_POS)} transplanted tokens · pads: p_kv {len(KV_PADS)} · p_dst {len(DST_PADS)}"
          f" · same positions: {KV_POS == DST_POS}")
    show_spans(dst_tok, f"p_dst · {tasks_str(DST_ORDER)} · {','.join(DST_DONE)} done · frontier {TN(FRONTIER)} · "
                        f"k/v PATCHED-TO (aligned) · pivot@{piv_dst}",
               task_span=DST_SPAN, kv_span=(DST_POS[0], DST_POS[-1] + 1), pad=DST_PADS)
    show_spans(kv_tok, f"p_kv · {tasks_str(KV_ORDER)} · kv-only expects {TN(KV_NEXT)} · k/v CACHED-FROM (aligned)",
               task_span=KV_SPAN, kv_span=(KV_POS[0], KV_POS[-1] + 1), pad=KV_PADS)

# ── PATCH_ATTN_RESTRICT: key positions the patch heads may still read on patched runs ──
if USE_KV:
    RESTRICT_KEEP = set(DST_POS)                                  # the window that holds p_kv's task-list k+v
else:
    RESTRICT_KEEP = set(range(*token_span(dst_prompt, task_text(DST_ORDER))))   # p_dst's own task list
RESTRICT_BLOCK = [q for q in range(piv_dst + 1) if q not in RESTRICT_KEEP and q != 0]   # pos 0 kept as a sink
if PATCH_ATTN_RESTRICT:
    print(f"PATCH_ATTN_RESTRICT: {len(PATCH_HEADS)} patch heads read {len(RESTRICT_KEEP)} task-span keys + pos 0; "
          f"{len(RESTRICT_BLOCK)} prompt keys masked")

# ── ATTN_ABL_SUFFICIENCY: non-goal heads in layers > l are blind from the task span through the pivot ──
_task_start = min(RESTRICT_KEEP)                              # p_kv window start (kv) / p_dst list start
ABL_BLOCK = list(range(max(_task_start - 1, 0), piv_dst + 1))  # as frontier_block_positions: span-1 .. end of prompt
_gh = set(map(tuple, PATCH_HEADS))
ABL_HEADS = [(l, h) for l in range(ATTN_ABL_LAYER + 1, mt.n_layer) for h in range(mt.n_heads) if (l, h) not in _gh]
if ATTN_ABL_SUFFICIENCY:
    print(f"ATTN_ABL_SUFFICIENCY: {len(ABL_HEADS)} non-goal heads in layers > {ATTN_ABL_LAYER} blinded from "
          f"{len(ABL_BLOCK)} keys (task span → pivot); {len(_gh)} goal heads spared")


pivot token: SRC ' =' · DST ' ='


╭─ SRC · {A,C,B} · A,B done · target C = 'count_capitalized' · pivot@127 ─────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the length of the longest word\n                                                                            │
│ (2) the number of capitalized words\n                                                                           │
│ (3) the number of distinct words\n                                                                              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = max(len(w) for w in words)\n                                                                            │
│     b = len(set(words))\n                                                                                       │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

align=pad: 35 transplanted tokens · pads: p_kv 5 · p_dst 0 · same positions: True


╭─ p_dst · {A,X,Y} · A done · frontier X = 'total_length' · k/v PATCHED-TO (aligned) · pivot@128 ─────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the length of the longest word\n                                                                            │
│ (2) the total number of characters across all words\n                                                           │
│ (3) the number of words that end with a consonant\n                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = max(len(w) for w in words)\n                                                                            │
│     b =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ p_kv · {A,B,D} · kv-only expects B = 'distinct_count' · k/v CACHED-FROM (aligned) ─────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ ⏎pad                                                                                                            │
│ ⏎pad                                                                                                            │
│ ⏎pad                                                                                                            │
│ ⏎pad                                                                                                            │
│ ⏎pad                                                                                                            │
│ (1) the length of the longest word\n                                                                            │
│ (2) the number of distinct words\n                                                                              │
│ (3) the total number of vowels across all words\n                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = max(len(w) for w in words)\n                                                                            │
│     b =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Runs — source / destination / (kv-only) / patched

| run | what runs | expect |
|---|---|---|
| source | clean p_src | C (SRC's target) |
| destination | clean, unpadded p_dst | DST's frontier |
| kv-only (`USE_KV`) | p_dst + p_kv's task-list k+v (+ boost) | p_kv's next (B) |
| patched | (kv +) SRC query at `PREFILL_ANCHORS`, re-clamped (`DECODE_Q`) for `N_STEER − 1` decode steps | `EXPECT_PATCHED` |

Each run: greedy generation + the GoalScope read-out ("what are you doing?") at its pivot. The patched read-out uses
the **prefill** patch only (`IFACE` = kv + the anchor q-patch + boost) — the decode re-clamps do not reach the pivot.

In [4]:
# ── source / destination / patched runs: generation + GoalScope read-out at the pivot ──
Q_LAYERS = sorted({l for l, _ in PATCH_HEADS})
_qh = heads_per_layer(PATCH_HEADS, "q")

# anchors = the last tokens of the prefill: "    c =" -> ['   ', ' c', ' ='] (indent, var, eq); indent pivot -> ['    ']
ANCHOR_OFFSET = {"indent": -3, "var": -2, "eq": -1} if PIVOT == "var_eq" else {"indent": -1}
assert set(PREFILL_ANCHORS) <= set(ANCHOR_OFFSET), f"PREFILL_ANCHORS {PREFILL_ANCHORS} need PIVOT='var_eq'"
anchor_pos = lambda tok: {a: tok["input_ids"].shape[1] + o for a, o in ANCHOR_OFFSET.items()}
SRC_ANCH, DST_ANCH = anchor_pos(src_tok), anchor_pos(dst_tok)
_tok_at = lambda tok, p: mt.tokenizer.decode([int(tok["input_ids"][0][p])])
for side, tok, anch in (("SRC", src_tok, SRC_ANCH), ("DST", dst_tok, DST_ANCH)):
    print(f"{side} anchors: " + "  ".join(f"{a}@{p}={_tok_at(tok, p)!r}" for a, p in anch.items()))
if PIVOT == "var_eq":
    assert all(_tok_at(t, a["indent"]).strip() == "" and _tok_at(t, a["eq"]).strip() == "="
               for t, a in ((src_tok, SRC_ANCH), (dst_tok, DST_ANCH))), "anchor offsets do not match the tokens"

# SRC queries (pre-RoPE) at every anchor: {layer: (n_anchors, width)}, rows in ANCHOR_OFFSET order
_anchors = list(ANCHOR_OFFSET)
q_src_all = cache_src_proj(src_prompt, [SRC_ANCH[a] for a in _anchors], Q_LAYERS, "q")
q_rows = lambda names: {l: c[[_anchors.index(a) for a in names]] for l, c in q_src_all.items()}

# prefill: row r of the cache goes to DST position r — indent->indent, var->var, eq->eq
PF = [dict(which="q", cache=q_rows(PREFILL_ANCHORS), heads=_qh,
           positions=[DST_ANCH[a] for a in PREFILL_ANCHORS])]
# decode: one query re-clamped at the current token for the first N_STEER-1 decode steps
if DECODE_Q == "eq" or PIVOT == "indent":
    q_dec = q_rows([_anchors[-1]])                                       # the " =" (or indent) query
elif DECODE_Q == "mean":
    q_dec = {l: c.mean(0, keepdim=True) for l, c in q_src_all.items()}  # mean of indent / var / eq
else:
    raise ValueError(f"DECODE_Q must be 'eq' or 'mean', got {DECODE_Q!r}")
DC = [dict(which="q", cache=q_dec, heads=_qh, positions=[-1])]
from contextlib import ExitStack


def patched_ctx(pf=None, boost=True):
    # stack: kv transplant (USE_KV) + query patch `pf` + ATTN_ABL_SUFFICIENCY + PATCH_ATTN_RESTRICT + the OV boost
    # (BOOST_ALPHA) — ONE patched forward
    s = ExitStack()
    if USE_KV:
        s.enter_context(headwise_kv_patch(mt, KV_HEADS, KV_K, KV_V, DST_POS))
    if pf:
        s.enter_context(proj_patch_hooks(mt, pf))
    if ATTN_ABL_SUFFICIENCY:             # sufficiency: everyone but the goal heads (layers > l) blind to task→pivot
        s.enter_context(restrict_head_attn(mt, ABL_HEADS, ABL_BLOCK, keep=[]))
    if PATCH_ATTN_RESTRICT:              # patch heads see only the task span (+ pos 0) of the prompt
        s.enter_context(restrict_head_attn(mt, PATCH_HEADS, RESTRICT_BLOCK, keep=[]))
    if boost and BOOST_ALPHA != 1.0:
        _bh = sorted(set(PATCH_HEADS) | (set(KV_HEADS) if USE_KV else set()))
        s.enter_context(boost_head_ov(mt, _bh, BOOST_ALPHA, positions=(-1,)))
    return s


def ro_iface(pf=None, boost=True):
    def _if(mt, inputs, patches=None, model_kwargs=None):
        with patched_ctx(pf, boost=boost):
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return _if


IFACE = ro_iface(PF)                       # kv (if USE_KV) + prefill q-patch (+ boost), for read-outs
EXPECT_PATCHED = (f"set+count {TN(H1_NEXT)}" if H1_NEXT == COUNT_NEXT
                  else f"set {TN(H1_NEXT)} / count {TN(COUNT_NEXT)}")      # what the patched run is judged against
print(f"prefill patch at {list(PREFILL_ANCHORS)} · decode query = {DECODE_Q if PIVOT == 'var_eq' else 'indent'} "
      f"· N_STEER={N_STEER}")

GEN, RO = {}, {}
GEN["source"] = greedy(src_tok, max_new=MAX_NEW)
rich_print(f"[red1]source · clean SRC (target {TN(TARGET)}):[/red1]\n{escape(GEN['source'])}")
RO["source"] = goalscope_readout("   ↳ goalscope · source pivot", src_prompt, piv_src, heads=READOUT_HEADS, **RO_KW)

GEN["destination"] = greedy(dst_tok_clean, max_new=MAX_NEW)
rich_print(f"[sea_green3]destination · clean DST (frontier {TN(FRONTIER)}):[/sea_green3]\n{escape(GEN['destination'])}")
RO["destination"] = goalscope_readout("   ↳ goalscope · destination pivot", dst_prompt, piv_dst_clean,
                                      heads=READOUT_HEADS, **RO_KW)

if USE_KV:
    with patched_ctx():
        GEN["kv"] = greedy(dst_tok, max_new=MAX_NEW)
    rich_print(f"[orange3]kv-only · DST + KV task-list k+v (expect {TN(KV_NEXT)}):[/orange3]\n{escape(GEN['kv'])}")
    RO["kv"] = goalscope_readout("   ↳ goalscope · kv-only pivot", dst_tok, piv_dst, heads=READOUT_HEADS,
                                 iface=ro_iface(), **RO_KW)

with patched_ctx():                        # kv + boost; the query patch itself comes from greedy's specs
    GEN["patched"] = greedy(dst_tok, prefill_specs=PF, decode_specs=DC, n_steer=N_STEER, max_new=MAX_NEW)
rich_print(f"[salmon1]patched · DST{' + KV k+v' if USE_KV else ''} + SRC query (expect {EXPECT_PATCHED}; "
           f"N_STEER={N_STEER}):[/salmon1]\n"
           f"{escape(GEN['patched'])}")
RO["patched"] = goalscope_readout(f"   ↳ goalscope · patched pivot (expect {EXPECT_PATCHED})", dst_tok, piv_dst,
                                  heads=READOUT_HEADS,
                                  iface=IFACE, **RO_KW)

SRC anchors: indent@125='   '  var@126=' c'  eq@127=' ='
DST anchors: indent@126='   '  var@127=' b'  eq@128=' ='
prefill patch at ['indent', 'var', 'eq'] · decode query = eq · N_STEER=32


source · clean SRC (target C = 'count_capitalized'):
 sum(1 for w in words if w.istitle())
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


   ↳ goalscope · source pivot  pos=127 ' =' => I'm analyzing the current line to determine if it's a unique 
variable name.

destination · clean DST (frontier X = 'total_length'):
 sum(len(w) for w in words)
    c = sum(1 for w in words if w[-1].lower() not in 'aeiou')
    return a, b, c
</code>

... (4 outputs left out)

   ↳ goalscope · kv-only pivot  pos=128 ' =' => I'm analyzing the current line of code to determine the number of 
unique elements.

patched · DST + KV k+v + SRC query (expect set D = 'total_vowels' / count Y = 'count_ends_consonant'; N_STEER=32):
 len(set(words))
    c = len(set(''.join(set(w) for w in words)))
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched pivot (expect set D = 'total_vowels' / count Y = 'count_ends_consonant')  pos=128 ' =' => 
I'm analyzing the current line of code to determine what specific action is being performed.

## Causality read-out — steer the pivot's goal encoding into an empty function

Same GoalScope steering, but instead of asking "what are you doing?", the goal-head encoding at the pivot is steered
into an information-less function stub and the model continues the code (mirrors `eval_goalscope --mode causality`,
whose stub is the C++ `bool check(auto value){`). The stub reuses DST's variable name and datatype and **always ends
at `return`**, whatever `PIVOT` is:
```
def func({var}: {type}):
    return
```
Read-outs for source, destination, kv-only (`USE_KV`) and patched (`IFACE`: kv + prefill q-patch + boost). The
continuation is the transferred *idea* — does the patched one implement the predicted task?

In [5]:
from src.goals.goalscope import apply_goalscope, GOALSCOPE_SYS

CAUSAL_MAX_NEW = 64
CAUSAL_INSTRUCTION_PY = (
    "You are implementing a simple Python function. Just write the function body, without any "
    "comments or explanations."
)
# Python stub from DST's own signature (same variable name + datatype as the destination prompt)
CAUSAL_PREFILL = DATA._signature(DATATYPE, dst_inst.var_name, "python") + "\n    return"


def _chat(user, assistant):
    msgs = [
        {"role": "system", "content": GOALSCOPE_SYS},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ]
    kw = dict(add_generation_prompt=False, continue_final_message=True, tokenize=False)
    try:
        return mt.tokenizer.apply_chat_template(
            msgs, chat_template_kwargs={"enable_thinking": False}, **kw
        )
    except TypeError:
        return mt.tokenizer.apply_chat_template(msgs, **kw)


CAUSAL_GS_PROMPT = _chat(CAUSAL_INSTRUCTION_PY, CAUSAL_PREFILL)


def causal_readout(tag, prompt, pivot, iface=None):
    res = apply_goalscope(
        prompt=prompt,
        mt=mt,
        selected_heads=list(READOUT_HEADS),
        strategy=RO_KW["strategy"],
        alpha=RO_KW["alpha"],
        steer_layer_range=RO_KW["layers"],
        # alpha=0.1,
        # steer_layer_range=(30, 41),
        batch_size=1,
        max_new_tokens=CAUSAL_MAX_NEW,
        goalscope_prompt=CAUSAL_GS_PROMPT,
        token_positions=[pivot],
        include_noncode=False,
        show=False,
        patch_interface=iface,
    )
    out = res[0]["patched_gen"]
    rich_print(
        f"[bold]{tag}[/bold]  pos={res[0]['pos']} {res[0]['token']!r} =>\n"
        f"[grey58]{escape(CAUSAL_PREFILL)}[/grey58][italic]{escape(out)}[/italic]"
    )
    return out


rich_print(f"[grey58]causal stub:[/grey58] {escape(CAUSAL_PREFILL)!s}")
CAUSAL = {
    "source": causal_readout(f"source (target {TN(TARGET)})", src_prompt, piv_src),
    "destination": causal_readout(
        f"destination (frontier {TN(FRONTIER)})", dst_prompt, piv_dst_clean
    ),
    "patched": causal_readout(
        f"patched{' (kv + q)' if USE_KV else ''} (expect {EXPECT_PATCHED})",
        dst_tok,
        piv_dst,
        iface=IFACE,
    ),
}
if USE_KV:
    CAUSAL["kv"] = causal_readout(
        f"kv-only (expect {TN(KV_NEXT)})", dst_tok, piv_dst, iface=ro_iface()
    )

causal stub: def func(words: list[str]):
    return

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

source (target C = 'count_capitalized')  pos=127 ' =' =>
def func(words: list[str]):
    return len(set(words))

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

destination (frontier X = 'total_length')  pos=128 ' =' =>
def func(words: list[str]):
    return sum(len(word) for word in words)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

patched (kv + q) (expect set D = 'total_vowels' / count Y = 'count_ends_consonant')  pos=128 ' =' =>
def func(words: list[str]):
    return len(set(words))

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

kv-only (expect B = 'distinct_count')  pos=128 ' =' =>
def func(words: list[str]):
    return len(set(words))

## Gaze — where does the pivot look in the task list?

Mean **value-weighted** attention from the pivot onto each listed task's description, averaged over `GAZE_HEADS`.
Measured under the plain patches — **no OV boost** (it scales the write, not the attention), as in 04.3.

| run | read over | patches |
|---|---|---|
| source | p_src's list | none |
| destination | p_dst's own list (clean, unpadded) | none |
| kv-only (`USE_KV`) | DST's task-list positions, as **p_kv's items** | kv |
| patched | DST's task-list positions — p_kv's items with kv, else DST's | (kv +) the **prefill** q-patch at `PREFILL_ANCHORS` |

With kv on, DST's task-list positions hold p_kv's key+value on the goal heads, so the per-task columns are p_kv's
tasks (mapped through the `pad` alignment) and the token-level plots show **p_kv's text**. The patched gaze shows
what the transplanted pivot query attends to; the decode re-clamps are not involved.

Plots: the table (argmax in bold), then token-level distributions — clean destination; kv-only; patched; and
patched − clean (without kv only, when both are over DST's own list).

In [6]:
def pivot_gaze(tok, items, piv, iface=None):
    # mean value-weighted pivot attention (GAZE_HEADS) per item; items = {letter: positions in `tok`}
    kw = {} if iface is None else dict(patch_interface=iface)
    info = get_attention_matrices(tok, mt, value_weighted=True, **kw)
    info = info[0] if isinstance(info, list) else info
    span = list(range(min(p[0] for p in items.values()), max(p[-1] for p in items.values()) + 1))
    a = pivot_attn(info, list(GAZE_HEADS), span, piv)                  # (len(span),)
    at = {p: v for p, v in zip(span, a)}
    per_item = {L: float(np.mean([at[p] for p in ps])) for L, ps in items.items()}
    return per_item, span, a

# gaze is measured under the plain patch (no OV boost — it scales the write, not the attention), as in 04.3
GAZE = {
    "source":      pivot_gaze(src_tok, item_positions(src_prompt, SRC_ORDER), piv_src),
    "destination": pivot_gaze(dst_tok_clean, item_positions(dst_prompt, DST_ORDER), piv_dst_clean),
}
if USE_KV:
    GAZE["kv"] = pivot_gaze(dst_tok, PATCHED_ITEMS, piv_dst, iface=ro_iface(boost=False))
GAZE["patched"] = pivot_gaze(dst_tok, PATCHED_ITEMS, piv_dst, iface=ro_iface(PF, boost=False))

t = Table(title=f"pivot gaze per task (mean value-weighted attn, {len(GAZE_HEADS)} heads) — argmax in bold"
                + (" · kv / patched read over p_kv's task span" if USE_KV else ""), highlight=False)
t.add_column("run"); t.add_column("expect")
for k in range(max(len(SRC_ORDER), len(DST_ORDER), len(KV_ORDER) if USE_KV else 0)):
    t.add_column(f"task {k + 1}", justify="right")
rows = [("source", TN(TARGET)), ("destination", TN(FRONTIER))]
rows += ([("kv", TN(KV_NEXT))] if USE_KV else []) + [("patched", EXPECT_PATCHED)]
for run, expect in rows:
    g_ = GAZE[run][0]; top = max(g_, key=g_.get)
    t.add_row(run, expect, *[f"[bold]{L}={v:.4f}[/bold]" if L == top else f"{L}={v:.4f}" for L, v in g_.items()])
rich_print(t)

# token-level views: clean DST over its own list; patched over DST's (kv-patched) task span
_, span_c, a_clean = GAZE["destination"]
toks_c = [mt.tokenizer.decode([int(dst_tok_clean["input_ids"][0][j])]) for j in span_c]
show_delta_tokens(toks_c, a_clean, "destination · pivot → DST tasks (clean)")
_, span, a_patched = GAZE["patched"]
toks = [patched_tok_text(j) for j in span]            # p_kv's task text when USE_KV
if USE_KV:
    show_delta_tokens(toks, GAZE["kv"][2], "kv-only · pivot → p_kv tasks")
show_delta_tokens(toks, a_patched, f"patched · pivot → {PATCHED_LABEL}")
# delta vs the run without the query: kv-only (same p_kv span) with kv, else the clean DST (same p_dst span)
_base_name, (_, span_b, a_base) = ("kv-only", GAZE["kv"]) if USE_KV else ("clean", GAZE["destination"])
assert span_b == span, "baseline and patched gaze must cover the same positions"
show_delta_tokens(toks, a_patched - a_base, f"patched − {_base_name} · pivot → {PATCHED_LABEL}")

 pivot gaze per task (mean value-weighted attn, 250 heads) — argmax in bold · kv / patched read over p_kv's 
                                                 task span                                                  
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┓
┃ run         ┃ expect                                                    ┃   task 1 ┃   task 2 ┃   task 3 ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━┩
│ source      │ C = 'count_capitalized'                                   │ A=0.0318 │ C=0.1913 │ B=0.1225 │
│ destination │ X = 'total_length'                                        │ A=0.0519 │ X=0.1763 │ Y=0.0535 │
│ kv          │ B = 'distinct_count'                                      │ A=0.0339 │ B=0.1051 │ D=0.0809 │
│ patched     │ set D = 'total_vowels' / count Y = 'count_ends_consonant' │ A=0.0254 │ B=0.0558 │ D=0.0939 │
└─────────────┴───────────────────────────────────────────────────────────┴──────────┴──────────┴──────────┘

destination · pivot → DST tasks (clean)   (green +Δ / red −Δ, |Δ|max=0.540)

kv-only · pivot → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.137)

patched · pivot → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.181)

patched − kv-only · pivot → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.088)

## Closed-loop query transfer — regenerate the goal heads' query at every step

The re-clamp run above writes the **same** SRC query into the current token at every decode step. Under the NRGS view
the goal persists but the query changes token by token ("given this goal and what I have written, what do I look
for next?"). The closed loop keeps the goal fixed and **recomputes** the query at every step:

1. **Prefill (i)** — the patched DST forward with the SRC queries at `PREFILL_ANCHORS` (plus kv + boost when
   `USE_KV`) → first token `t1`.
2. **Goal encoding (ii)** — the goal heads' OV write (`READOUT_HEADS`) **in that same patched prefill**, at ` =`
   (`GOAL_ENC="eq"`) or averaged over `PREFILL_ANCHORS` (`"mean"`). Never from the clean SRC. Fixed for the whole loop. `GOAL_ENC_CONTRAST=True` injects patched − base instead, where base is the same
   run without the SRC query (kv-only with kv, else the clean DST); compare all three in the read-outs cell first.
   `GOAL_ENC_RUN="restricted"` reads it instead from ONE separate forward: kv (if `USE_KV`) + the SRC query at
   `PREFILL_ANCHORS`, with the goal heads attending only to the task span (+ pos 0), no boost.
3. **Probe** (`PROBE_PROMPT`; default `"stub"`) — `"tasklist"` adds the task list the goal heads read (p_kv's
   with kv, else p_dst's) to the user turn; `"self"` uses the (aligned) DST prompt itself, kv-patched if `USE_KV`,
   with no query patch / boost / restriction. `"gaze_select"` names only the task the patched pivot gazes at most
   (`GAZE["patched"]` argmax) in the user turn, with no goal injection unless `GAZE_SELECT_INJECT=True`. The stub is an information-less prompt with DST's signature, ending at the bare trailing indent
   (`PIVOT="indent"`) or `    return` (`PIVOT="var_eq"`). The goal encoding is steered in with `additive_norm`
   (α `PROBE_ALPHA`, residual after layers `PROBE_LAYERS` = 35–45) at the last `|PROBE_STEER_START|` prompt tokens
   (default −1 = the pivot only; −5 = the last 5 prompt tokens incl. the pivot) and, with `PROBE_STEER="all"`, at
   every appended token (`"pivot"`: the prompt tokens only). The probe is never kv- or q-patched.
4. **Each step k ≥ 1 (iii)** — append `t_k` to the probe; read its query at `t_k` on the goal heads (`q_caus`);
   write `q_caus` into the patched run's goal-head query at `t_k` (every other head keeps its own; kv + boost stay
   on); generate `t_{k+1}`. Stop at the first newline or `LOOP_MAX` tokens.

So the goal heads' query comes from SRC at the prefill anchors, and from the steered probe at every generated token.
Output: a per-step table (`t_k` and cos(`q_caus`, the re-clamp run's fixed decode query), mean over heads) and the
first line of the closed loop next to the clean destination and the re-clamp run.

In [24]:
# ── knobs ────────────────────────────────────────────────────────────────────
GOAL_ENC_RUN = "restricted"   # where the goal encoding is read: "patched" = the patched prefill (IFACE: kv + q +
                           # PATCH_ATTN_RESTRICT + boost) · "restricted" = ONE separate forward: kv (if USE_KV) + the
                           # SRC query at PREFILL_ANCHORS, PATCH_HEADS attending only to the task span (p_kv window
                           # with kv, else p_dst list) + pos 0; no OV boost; independent of PATCH_ATTN_RESTRICT
ENC_WHITELIST_ANCHORS = True   # restricted run: also let the goal heads read the last prompt tokens = the anchors
                               # (indent, var, eq under PIVOT="var_eq"; the indent otherwise) besides the task span
GOAL_ENC_CONTRAST = False  # True: inject g_patched − g_base, base = the SAME run without the SRC query (kv-only with
                           # kv, else clean DST) — removes what the pivot shares with the native run. Check the
                           # encoding read-outs cell first.
PROBE_PROMPT = "gaze_select"      # "stub": information-less signature stub · "tasklist": the stub + the task list the goal heads
                           # read (p_kv's with kv, else p_dst's) in the user turn · "self": the (aligned) DST prompt itself,
                           # kv-patched if USE_KV, no q-patch / boost / restriction — all steered with the goal encoding
                           # "gaze_select": the stub whose user turn names ONLY the task the patched pivot gazes at
                           # most (argmax of GAZE["patched"]; p_kv's items with kv) — no goal injection unless
GAZE_SELECT_INJECT = True # True: also inject the goal encoding into the "gaze_select" probe
GOAL_ENC    = "eq"         # "eq": goal heads' OV write at DST " =" · "mean": mean over PREFILL_ANCHORS
PROBE_STEER = "all"        # "pivot": steer only the probe's " return" · "all": also every appended token
PROBE_STEER_START = -20     # first probe PROMPT token to steer, relative to the end: -1 = the pivot only (default),
                           # -5 = the last 5 prompt tokens incl. the pivot (all before the loop starts)
LOOP_MAX    = 25           # max generated tokens (also stops at the first newline)
PROBE_ALPHA, PROBE_LAYERS = 0.1, (20, 40)

import baukit
from contextlib import contextmanager, nullcontext
from src.goals.goalscope import GOALSCOPE_SYS
from src.goals.head_loc_attn import collect_head_steering_vecs

# (2) goal encoding — goal heads' OV write at the DST anchors, read from GOAL_ENC_RUN
_ENC_KEEP = set(DST_ANCH.values()) if ENC_WHITELIST_ANCHORS else set()       # anchor positions (aligned DST)
ENC_BLOCK = [q for q in RESTRICT_BLOCK if q not in _ENC_KEEP]                   # keys masked in the restricted run


def _restricted_enc_iface(mt, inputs, patches=None, model_kwargs=None):
    with ExitStack() as s:
        if USE_KV:
            s.enter_context(headwise_kv_patch(mt, KV_HEADS, KV_K, KV_V, DST_POS))
        s.enter_context(proj_patch_hooks(mt, PF))
        s.enter_context(restrict_head_attn(mt, PATCH_HEADS, ENC_BLOCK, keep=[]))
        return mt._model(**dict(inputs), **dict(model_kwargs or {}))


if GOAL_ENC_RUN == "restricted":
    print(f"restricted goal-encoding run: goal heads read the task span + pos 0"
          + (f" + anchors {[ _tok_at(dst_tok, q) for q in sorted(_ENC_KEEP)]}" if _ENC_KEEP else "")
          + f" · {len(ENC_BLOCK)} prompt keys masked")
if GOAL_ENC_RUN not in ("patched", "restricted"):
    raise ValueError(f"GOAL_ENC_RUN must be 'patched' or 'restricted', got {GOAL_ENC_RUN!r}")
def _restricted_base_iface(mt, inputs, patches=None, model_kwargs=None):   # the restricted run WITHOUT the query
    with ExitStack() as s:
        if USE_KV:
            s.enter_context(headwise_kv_patch(mt, KV_HEADS, KV_K, KV_V, DST_POS))
        s.enter_context(restrict_head_attn(mt, PATCH_HEADS, ENC_BLOCK, keep=[]))
        return mt._model(**dict(inputs), **dict(model_kwargs or {}))


_anch = list(PREFILL_ANCHORS)
if GOAL_ENC not in ("eq", "mean"):
    raise ValueError(f"GOAL_ENC must be 'eq' or 'mean', got {GOAL_ENC!r}")
assert GOAL_ENC != "eq" or "eq" in _anch, "GOAL_ENC='eq' needs 'eq' in PREFILL_ANCHORS"


def _enc(iface):
    ov = collect_head_steering_vecs(mt=mt, src_tokenized=dst_tok, selected_heads=list(READOUT_HEADS),
                                    token_positions=[DST_ANCH[a] for a in _anch], patch_interface=iface)[0]
    return ov[_anch.index("eq")] if GOAL_ENC == "eq" else ov.mean(0)


_base_iface = ro_iface() if GOAL_ENC_RUN == "patched" else _restricted_base_iface   # same run, no SRC query
ENC = {"patched": _enc(IFACE if GOAL_ENC_RUN == "patched" else _restricted_enc_iface), "base": _enc(_base_iface)}
ENC["contrast"] = ENC["patched"] - ENC["base"]
_g = ENC["contrast"] if GOAL_ENC_CONTRAST else ENC["patched"]
G_UNIT = (_g / _g.norm()).float()                     # additive_norm: unit direction, magnitude from ||h||
_cosb = float(torch.nn.functional.cosine_similarity(ENC["patched"].float(), ENC["base"].float(), dim=0))
print(f"goal encoding: {'contrast (patched − base)' if GOAL_ENC_CONTRAST else 'patched'} · cos(patched, base) = "
      f"{_cosb:.3f} · base = {'kv-only' if USE_KV else 'clean DST'} ({GOAL_ENC_RUN} run without the SRC query)")

if GOAL_ENC_RUN == "restricted":
    # pivot gaze of the restricted goal-encoding run, over the task span the goal heads may read
    # (p_kv's items inside DST with kv, else p_dst's); value-weighted, GAZE_HEADS — as in the gaze section
    _enc_items, _enc_span, _enc_a = pivot_gaze(dst_tok, PATCHED_ITEMS, piv_dst, iface=_restricted_enc_iface)
    _top = max(_enc_items, key=_enc_items.get)
    rich_print(f"[bold]restricted goal-encoding run · pivot gaze per task[/bold] ({len(GAZE_HEADS)} heads; expect "
               f"{EXPECT_PATCHED}): " + "  ".join(f"[bold]{L}={v:.4f}[/bold]" if L == _top else f"{L}={v:.4f}"
                                              for L, v in _enc_items.items()))
    show_delta_tokens([patched_tok_text(j) for j in _enc_span], _enc_a,
                      f"restricted goal-encoding run · pivot → {PATCHED_LABEL}")

# probe: information-less function stub, same variable name + datatype as DST
# pivot shape follows PIVOT: "indent" -> bare trailing indent (else the model writes `return b = ...`),
# "var_eq" -> `return`
PROBE_PREFILL = DATA._signature(DATATYPE, dst_inst.var_name, "python") + ("\n    return" if PIVOT == "var_eq" else "\n    ")
PROBE_INSTRUCTION = ("You are implementing a simple Python function. Just write the function body, without any "
                     "comments or explanations.")


def _probe_chat(user, assistant):
    msgs = [{"role": "system", "content": GOALSCOPE_SYS}, {"role": "user", "content": user},
            {"role": "assistant", "content": assistant}]
    kw = dict(add_generation_prompt=False, continue_final_message=True, tokenize=False)
    try:
        return mt.tokenizer.apply_chat_template(msgs, chat_template_kwargs={"enable_thinking": False}, **kw)
    except TypeError:
        return mt.tokenizer.apply_chat_template(msgs, **kw)


if PROBE_PROMPT == "stub":
    probe_prompt = _probe_chat(PROBE_INSTRUCTION, PROBE_PREFILL)
elif PROBE_PROMPT == "tasklist":
    # the task question of the list the goal heads read (p_kv's with kv, else p_dst's) + the same stub
    _list_inst = kv_inst if USE_KV else dst_inst
    probe_prompt = _probe_chat(_list_inst.dst_parts()[0].strip() + "\n" + PROBE_INSTRUCTION, PROBE_PREFILL)
elif PROBE_PROMPT == "self":
    probe_prompt, PROBE_PREFILL = None, "(the DST prompt itself)"
elif PROBE_PROMPT == "gaze_select":
    # the task the PATCHED pivot attends to most (gaze section, prefill q-patch, no boost) names the sub-goal
    _pg = GAZE["patched"][0]
    GAZE_SELECTED = max(_pg, key=_pg.get)
    rich_print(f"gaze_select: patched pivot gaze " + "  ".join(f"{L}={v:.4f}" for L, v in _pg.items())
               + f" -> [bold]{TN(GAZE_SELECTED)}[/bold] (expect {EXPECT_PATCHED})")
    probe_prompt = _probe_chat(f"You are implementing a simple Python function that computes "
                               f"{by[TASKS[GAZE_SELECTED]].description}. Just write the function body, without any "
                               f"comments or explanations.", PROBE_PREFILL)
else:
    raise ValueError(f"PROBE_PROMPT must be 'stub', 'tasklist', 'self' or 'gaze_select', got {PROBE_PROMPT!r}")

print(probe_prompt)

PROBE_INJECT = not (PROBE_PROMPT == "gaze_select" and not GAZE_SELECT_INJECT)   # goal injection on the probe?
probe_tok = dst_tok if PROBE_PROMPT == "self" else prepare_input(prompts=probe_prompt, tokenizer=mt)
piv_probe = probe_tok["input_ids"].shape[1] - 1


def probe_ctx():
    # the self-probe sees DST exactly as the patched run does, minus the query patch: kv (if USE_KV) only
    if PROBE_PROMPT == "self" and USE_KV:
        return headwise_kv_patch(mt, KV_HEADS, KV_K, KV_V, DST_POS)
    return nullcontext()
assert -probe_tok["input_ids"].shape[1] <= PROBE_STEER_START <= -1, "PROBE_STEER_START must be in [-len(probe), -1]"
PROBE_STEER_POS = list(range(piv_probe + 1 + PROBE_STEER_START, piv_probe + 1))   # prompt positions steered at prefill
print(f"probe = {PROBE_PROMPT} (goal injection {'on' if PROBE_INJECT else 'OFF'}) · probe pivot {mt.tokenizer.decode(probe_tok['input_ids'][0][piv_probe:piv_probe + 1])!r} · steered prompt "
      f"tokens {[mt.tokenizer.decode([int(probe_tok['input_ids'][0][q])]) for q in PROBE_STEER_POS]} · "
      f"goal enc = {GOAL_ENC} over {_anch} from the {GOAL_ENC_RUN} run · probe steer = {PROBE_STEER} · layers {PROBE_LAYERS}")

_STEER_NAMES = [mt.layer_name_format.format(l) for l in range(*PROBE_LAYERS)]
_Q_NAMES = {l: FUNC._proj_name(l, "q", mt) for l in Q_LAYERS}


@contextmanager
def goal_steer(positions):
    # h <- h + alpha * ||h|| * g_unit at `positions` of the current forward, residual after each steer layer
    # (a no-op when PROBE_INJECT is off: the gaze_select probe is conditioned by its text alone)
    if not PROBE_INJECT:
        yield
        return
    def edit(output, layer):
        h = output[0] if isinstance(output, tuple) else output
        for p in positions:
            v = h[:, p, :]
            h[:, p, :] = v + PROBE_ALPHA * v.float().norm(dim=-1, keepdim=True).to(v.dtype) \
                * G_UNIT.to(v.device, v.dtype)
        return output
    with baukit.TraceDict(mt._model, _STEER_NAMES, edit_output=edit):
        yield


def _ext(am):
    return torch.cat([am, torch.ones_like(am[:, :1])], 1)


def _head_cos(qa, qb):
    # mean over PATCH_HEADS of cos(q_a, q_b) per head slice
    hd, out = mt.head_dim, []
    for l, h in PATCH_HEADS:
        a, b = qa[l][0, h * hd:(h + 1) * hd], qb[l][0, h * hd:(h + 1) * hd]
        out.append(float(torch.nn.functional.cosine_similarity(a, b, dim=0)))
    return float(np.mean(out))


@torch.no_grad()
def closed_loop(verbose=True):
    eos = mt.tokenizer.eos_token_id
    # (1) patched-run prefill: SRC queries at the anchors -> t1
    ids, am = dst_tok["input_ids"], dst_tok["attention_mask"]
    with patched_ctx(pf=PF):
        out = mt._model(input_ids=ids, attention_mask=am, use_cache=True)
    pkv_d, nxt = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
    # probe prefill: goal steered at the last |PROBE_STEER_START| prompt tokens (the pivot by default)
    p_am = probe_tok["attention_mask"]
    with probe_ctx(), goal_steer(PROBE_STEER_POS):
        pout = mt._model(input_ids=probe_tok["input_ids"], attention_mask=p_am, use_cache=True)
    pkv_p = pout.past_key_values

    gen, steps = [int(nxt)], []
    while len(gen) < LOOP_MAX and int(nxt) != eos and "\n" not in mt.tokenizer.decode(gen):
        # (3) probe consumes t_k (goal-steered if PROBE_STEER="all"); read q_caus at t_k on the goal heads
        p_am = _ext(p_am)
        ctx = goal_steer([-1]) if PROBE_STEER == "all" else nullcontext()
        with probe_ctx(), ctx, baukit.TraceDict(mt._model, list(_Q_NAMES.values()), retain_output=True) as tr:
            pout = mt._model(input_ids=nxt, attention_mask=p_am, past_key_values=pkv_p, use_cache=True)
        pkv_p = pout.past_key_values
        q_caus = {l: tr[n].output[0, -1:].detach().float().cpu() for l, n in _Q_NAMES.items()}
        # patched run: q_caus replaces the goal heads' query at t_k -> t_{k+1}
        am = _ext(am)
        with patched_ctx(pf=[dict(which="q", cache=q_caus, heads=_qh, positions=[-1])]):
            out = mt._model(input_ids=nxt, attention_mask=am, past_key_values=pkv_d, use_cache=True)
        pkv_d = out.past_key_values
        steps.append(dict(k=len(gen), tok=mt.tokenizer.decode(gen[-1:]), cos_q_dec=_head_cos(q_caus, q_dec), q=q_caus))
        nxt = out.logits[:, -1].argmax(-1, keepdim=True)
        gen.append(int(nxt))
    text = mt.tokenizer.decode(gen, skip_special_tokens=True)
    if verbose:
        t = Table(title=f"closed loop · per step: token t_k whose query is replaced · cos(q_caus, decode q of the "
                        f"re-clamp run), mean over {len(PATCH_HEADS)} heads", highlight=False)
        t.add_column("k", justify="right"); t.add_column("t_k"); t.add_column("cos", justify="right")
        for s in steps:
            t.add_row(str(s["k"]), repr(s["tok"]), f"{s['cos_q_dec']:.3f}")
        rich_print(t)
    return text, steps, gen


restricted goal-encoding run: goal heads read the task span + pos 0 + anchors ['   ', ' b', ' ='] · 90 prompt keys masked
goal encoding: patched · cos(patched, base) = 0.780 · base = kv-only (restricted run without the SRC query)


restricted goal-encoding run · pivot gaze per task (250 heads; expect set D = 'total_vowels' / count Y = 
'count_ends_consonant'): A=0.0342  B=0.0949  D=0.1810

restricted goal-encoding run · pivot → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.442)

gaze_select: patched pivot gaze A=0.0254  B=0.0558  D=0.0939 -> D = 'total_vowels' (expect set D = 'total_vowels' /
count Y = 'count_ends_consonant')

<|im_start|>system
You are a helpful assistant proficient in programming.<|im_end|>
<|im_start|>user
You are implementing a simple Python function that computes the total number of vowels across all words. Just write the function body, without any comments or explanations.<|im_end|>
<|im_start|>assistant
def func(words: list[str]):
    return
probe = gaze_select (goal injection on) · probe pivot ' return' · steered prompt tokens [' without', ' any', ' comments', ' or', ' explanations', '.', '<|im_end|>', '\n', '<|im_start|>', 'assistant', '\n', 'def', ' func', '(words', ':', ' list', '[str', ']):\n', '   ', ' return'] · goal enc = eq over ['indent', 'var', 'eq'] from the restricted run · probe steer = all · layers (20, 40)


### Goal-encoding read-outs — patched vs base vs contrast (check before injecting)

Faithfulness (GoalScope "what are you doing?") and causality (the `return` stub of the causality section) read-outs of
the three candidate vectors, with the paper read-out config (`RO_KW`):
* **patched** — the goal heads' OV write at the DST anchors in the `GOAL_ENC_RUN` run (with the SRC query);
* **base** — the same run *without* the SRC query (kv-only with kv, else the clean DST);
* **contrast** — patched − base (`GOAL_ENC_CONTRAST=True` injects this one).

In [25]:
from src.goals.goalscope import head_additive_generate, _build_goalscope_prompt

_enc_ro = lambda vec, prompt: head_additive_generate(
    mt=mt,
    dest_inputs=prepare_input(prompts=prompt, tokenizer=mt),
    steering_vecs=vec.float(),
    # layers=list(range(*RO_KW["layers"])), alpha=RO_KW["alpha"],
    layers=list(range(30, 40)), alpha=0.1,
    max_new_tokens=64,
    strategy=RO_KW["strategy"],
)
ENC_RO = {}
for name, vec in ENC.items():
    ENC_RO[name] = dict(
        faith=_enc_ro(vec, _build_goalscope_prompt(mt)),
        causal=_enc_ro(vec, CAUSAL_GS_PROMPT),
    )
    tag = (
        "  ← injected"
        if name == ("contrast" if GOAL_ENC_CONTRAST else "patched")
        else ""
    )
    rich_print(
        f"[bold]{name}[/bold]{tag}  (expect {EXPECT_PATCHED})\n"
        f"  faithfulness => [italic]{escape(ENC_RO[name]['faith'])}[/italic]\n"
        f"  causality    => [grey58]{escape(CAUSAL_PREFILL)}[/grey58][italic]{escape(ENC_RO[name]['causal'])}[/italic]"
    )

patched  ← injected  (expect set D = 'total_vowels' / count Y = 'count_ends_consonant')
  faithfulness => I'm analyzing the specific line of code to determine its functionality or content.
  causality    => def func(words: list[str]):
    return len(set(words))

base  (expect set D = 'total_vowels' / count Y = 'count_ends_consonant')
  faithfulness => I'm analyzing the current line of code to determine its specific functionality or value.
  causality    => def func(words: list[str]):
    return len(set(words))

contrast  (expect set D = 'total_vowels' / count Y = 'count_ends_consonant')
  faithfulness => I'm analyzing the specific line of code you're referring to. Please provide the line for context.
  causality    => def func(words: list[str]):
    return [word.upper() for word in words]

### Run the closed loop

In [27]:
GEN["closed_loop"], LOOP_STEPS, LOOP_IDS = closed_loop()
_line = lambda s: s.split("\n")[0]
rich_print(f"[sea_green3]destination (frontier {TN(FRONTIER)}):[/sea_green3] {escape(_line(GEN['destination']))}")
rich_print(f"[salmon1]q_src re-clamp (N_STEER={N_STEER}):[/salmon1]   {escape(_line(GEN['patched']))}")
rich_print(f"[bold magenta]closed loop (expect {EXPECT_PATCHED}):[/bold magenta]  {escape(_line(GEN['closed_loop']))}")

closed loop · per step: 
token t_k whose query is
 replaced · cos(q_caus, 
decode q of the re-clamp
  run), mean over 250   
         heads          
┏━━━┳━━━━━━━━━━┳━━━━━━━┓
┃ k ┃ t_k      ┃   cos ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━┩
│ 1 │ ' len'   │ 0.668 │
│ 2 │ '(set'   │ 0.642 │
│ 3 │ '(words' │ 0.613 │
└───┴──────────┴───────┘

destination (frontier X = 'total_length'):  sum(len(w) for w in words)

q_src re-clamp (N_STEER=32):    len(set(words))

closed loop (expect set D = 'total_vowels' / count Y = 'count_ends_consonant'):   len(set(words))

### Closed loop — read-outs of the goal encoding (ii) + gaze along the loop

* **Read-outs of the goal encoding itself** (`_g`, what drives `q_caus`): GoalScope ("what are you doing?") and a
  causality read-out on the **loop's probe prompt** (bare indent under `PIVOT="indent"`, `return` under
  `"var_eq"`), steered as in the loop (`PROBE_STEER`). Under `PIVOT="indent"` this prompt differs from the causality
  section's `return` stub, so the two continuations need not match even though the vector is the same.
* **Gaze of the patched run** at the pivot (prefill patch only) and at every loop token `t_k` (its goal-head query =
  `q_caus`), over the same positions as the gaze section (p_kv's items when `USE_KV`). Obtained by replaying
  DST + `t_1..t_m` with the identical patches (kv, prefill anchors, `q_caus` at each `t_k`; no boost), which
  reproduces the loop's forward. Plots: the per-step table, then the **actual** attention distribution at every
  `t_k`.

In [28]:
from src.goals.goalscope import head_additive_generate, _build_goalscope_prompt
from src.utils.typing import TokenizerOutput

# ── read-outs of the goal encoding ───────────────────────────────────────────
_ro = lambda prompt, n_steer: head_additive_generate(
    mt=mt, dest_inputs=prepare_input(prompts=prompt, tokenizer=mt), steering_vecs=_g.float(),
    layers=list(range(*PROBE_LAYERS)), max_new_tokens=64, strategy="additive_norm", n_steer_tokens=n_steer,
    alpha=PROBE_ALPHA)
@torch.no_grad()
def probe_free_run(max_new=64):
    # the loop's probe, steered exactly as in the loop (PROBE_STEER_POS at prefill, + every token if "all"),
    # but free-running on its own tokens — what the goal encoding alone writes
    p_am = probe_tok["attention_mask"]
    with probe_ctx(), goal_steer(PROBE_STEER_POS):
        out = mt._model(input_ids=probe_tok["input_ids"], attention_mask=p_am, use_cache=True)
    pkv, nxt, gen = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True), []
    while len(gen) < max_new and int(nxt) != mt.tokenizer.eos_token_id:
        gen.append(int(nxt)); p_am = _ext(p_am)
        with probe_ctx(), (goal_steer([-1]) if PROBE_STEER == "all" else nullcontext()):
            out = mt._model(input_ids=nxt, attention_mask=p_am, past_key_values=pkv, use_cache=True)
        pkv, nxt = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
    return mt.tokenizer.decode(gen, skip_special_tokens=True)


LOOP_RO = {
    "goalscope": _ro(_build_goalscope_prompt(mt), None),
    "causality": probe_free_run(),
}
rich_print(f"[bold]goal encoding ({GOAL_ENC}) · GoalScope[/bold] => [italic]{escape(LOOP_RO['goalscope'])}[/italic]")
rich_print(f"[bold]goal encoding ({GOAL_ENC}) · causality[/bold] =>\n[grey58]{escape(PROBE_PREFILL)}[/grey58]"
           f"[italic]{escape(LOOP_RO['causality'])}[/italic]")

# ── gaze along the loop ─────────────────────────────────────────────────────
L0 = dst_tok["input_ids"].shape[1]
m = len(LOOP_STEPS)        # steps whose query was replaced: t_1 .. t_m
replay = TokenizerOutput(data=dict(
    input_ids=torch.cat([dst_tok["input_ids"], torch.tensor([LOOP_IDS[:m]], device=dst_tok["input_ids"].device)], 1),
    attention_mask=torch.ones(1, L0 + m, dtype=dst_tok["attention_mask"].dtype,
                              device=dst_tok["attention_mask"].device)))
loop_spec = dict(which="q", heads=_qh, positions=[L0 + k for k in range(m)],
                 cache={l: torch.cat([s["q"][l] for s in LOOP_STEPS], 0) for l in Q_LAYERS})
_info = get_attention_matrices(replay, mt, value_weighted=True,
                               patch_interface=ro_iface(PF + [loop_spec], boost=False))
_info = _info[0] if isinstance(_info, list) else _info

_items = PATCHED_ITEMS                    # DST items (or KV items when USE_KV)
_span = list(range(min(p[0] for p in _items.values()), max(p[-1] for p in _items.values()) + 1))
def _per_item(a):
    at = dict(zip(_span, a)); return {L: float(np.mean([at[p] for p in ps])) for L, ps in _items.items()}

LOOP_GAZE = [("pivot " + repr(mt.tokenizer.decode(dst_tok["input_ids"][0][piv_dst:piv_dst + 1])),
              pivot_attn(_info, list(GAZE_HEADS), _span, piv_dst))]
LOOP_GAZE += [(f"t_{s['k']} {s['tok']!r}", pivot_attn(_info, list(GAZE_HEADS), _span, L0 + s["k"] - 1))
              for s in LOOP_STEPS]
t = Table(title=f"closed-loop gaze over {PATCHED_LABEL} (mean value-weighted attn, {len(GAZE_HEADS)} heads; frontier "
                f"{TN(FRONTIER)}, SRC target {TN(TARGET)}" + (f", kv-only {TN(KV_NEXT)}" if USE_KV else "") + f", expect {EXPECT_PATCHED}" + ") — argmax in bold", highlight=False)
t.add_column("query"); [t.add_column(L, justify="right") for L in _items]
for name, a in LOOP_GAZE:
    g_ = _per_item(a); top = max(g_, key=g_.get)
    t.add_row(name, *[f"[bold]{v:.4f}[/bold]" if L == top else f"{v:.4f}" for L, v in g_.items()])
rich_print(t)

_toks = [patched_tok_text(j) for j in _span]           # p_kv's task text when USE_KV
# actual gaze (not delta) of the goal heads at every loop step t_k, whose query was replaced by q_caus
for name, a in LOOP_GAZE[1:]:
    show_delta_tokens(_toks, a, f"closed loop · {name} (query = q_caus) → {PATCHED_LABEL}")

goal encoding (eq) · GoalScope => I'm analyzing the number of unique characters in the string.

goal encoding (eq) · causality =>
def func(words: list[str]):
    return 
len(set(len(set(''.join(''.join(set(map(len(set(''.join(''.join(set(map(len(set(''.join(''.join(set(''.join(''.join
(set(''.join(''.join(set(''.join(''.join(set(''.join(''.join(set(''.join(''.

  closed-loop gaze over p_kv tasks (mean   
value-weighted attn, 250 heads; frontier X 
     = 'total_length', SRC target C =      
     'count_capitalized', kv-only B =      
     'distinct_count', expect set D =      
        'total_vowels' / count Y =         
 'count_ends_consonant') — argmax in bold  
┏━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃ query        ┃      A ┃      B ┃      D ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
│ pivot ' ='   │ 0.0253 │ 0.0558 │ 0.0940 │
│ t_1 ' len'   │ 0.0219 │ 0.0188 │ 0.0333 │
│ t_2 '(set'   │ 0.0214 │ 0.0202 │ 0.0328 │
│ t_3 '(words' │ 0.0245 │ 0.0249 │ 0.0320 │
└──────────────┴────────┴────────┴────────┘

closed loop · t_1 ' len' (query = q_caus) → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.054)

closed loop · t_2 '(set' (query = q_caus) → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.055)

closed loop · t_3 '(words' (query = q_caus) → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.063)

### Re-clamp run — gaze along the generation (compare with the closed loop)

Replays the **re-clamp** patched run (SRC query at `PREFILL_ANCHORS`, then `DECODE_Q` re-clamped for the first
`N_STEER − 1` decode steps, kv if `USE_KV`) for as many tokens as the closed loop, and shows the goal heads' gaze at the
pivot and at every generated token — over the same positions as the loop gaze. If the re-clamp gaze also falls back
to the native task after the pivot while the re-clamp still writes the target, the post-pivot gaze is not the signal
that decides the outcome.

In [29]:
# @torch.no_grad()
# def reclamp_ids(m):
#     # the re-clamp patched run (as GEN["patched"]), token ids, for m tokens or until the first newline
#     ids, am = dst_tok["input_ids"], dst_tok["attention_mask"]
#     with patched_ctx(pf=PF):
#         out = mt._model(input_ids=ids, attention_mask=am, use_cache=True)
#     pkv, nxt = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
#     gen = [int(nxt)]
#     for s in range(1, m):
#         if int(nxt) == mt.tokenizer.eos_token_id or "\n" in mt.tokenizer.decode(gen):
#             break
#         am = _ext(am)
#         with patched_ctx(pf=DC if s < N_STEER else None):
#             out = mt._model(input_ids=nxt, attention_mask=am, past_key_values=pkv, use_cache=True)
#         pkv, nxt = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
#         gen.append(int(nxt))
#     return gen


# RC_IDS = reclamp_ids(LOOP_MAX)
# m_rc = len(RC_IDS) - 1                         # tokens whose query is re-clamped / read: t_1 .. t_{m}
# rc_pos = [L0 + s - 1 for s in range(1, m_rc + 1)]
# rc_steered = [q for s, q in zip(range(1, m_rc + 1), rc_pos) if s < N_STEER]
# replay_rc = TokenizerOutput(data=dict(
#     input_ids=torch.cat([dst_tok["input_ids"], torch.tensor([RC_IDS[:m_rc]], device=dst_tok["input_ids"].device)], 1),
#     attention_mask=torch.ones(1, L0 + m_rc, dtype=dst_tok["attention_mask"].dtype,
#                               device=dst_tok["attention_mask"].device)))
# rc_spec = [dict(which="q", heads=_qh, positions=rc_steered,
#                 cache={l: c.repeat(len(rc_steered), 1) for l, c in q_dec.items()})] if rc_steered else []
# _info_rc = get_attention_matrices(replay_rc, mt, value_weighted=True, patch_interface=ro_iface(PF + rc_spec, boost=False))
# _info_rc = _info_rc[0] if isinstance(_info_rc, list) else _info_rc

# RC_GAZE = [("pivot " + repr(_tok_at(dst_tok, piv_dst)), pivot_attn(_info_rc, list(GAZE_HEADS), _span, piv_dst))]
# RC_GAZE += [(f"t_{s} {mt.tokenizer.decode(RC_IDS[s - 1:s])!r}", pivot_attn(_info_rc, list(GAZE_HEADS), _span, q))
#             for s, q in zip(range(1, m_rc + 1), rc_pos)]
# rich_print(f"[salmon1]re-clamp ({m_rc + 1} tok):[/salmon1] {escape(mt.tokenizer.decode(RC_IDS))!s}")
# t = Table(title=f"re-clamp gaze over {PATCHED_LABEL} (expect {EXPECT_PATCHED}; query re-clamped for t_1..t_"
#                 f"{min(m_rc, N_STEER - 1)}) — argmax in bold", highlight=False)
# t.add_column("query"); [t.add_column(L, justify="right") for L in _items]
# for name, a in RC_GAZE:
#     g_ = _per_item(a); top = max(g_, key=g_.get)
#     t.add_row(name, *[f"[bold]{v:.4f}[/bold]" if L == top else f"{v:.4f}" for L, v in g_.items()])
# rich_print(t)
# for name, a in RC_GAZE[1:]:
#     show_delta_tokens(_toks, a, f"re-clamp · {name} → {PATCHED_LABEL}")

## Selection probe — let the goal encoding pick the sub-goal (k-way), then run the closed loop

Instead of the pivot gaze (`PROBE_PROMPT="gaze_select"`), the **goal encoding** chooses the task. A k-way prompt lists
the tasks of the patched list (p_kv's with kv, else p_dst's — the written one included) and asks for one:
```
<user> Select one of the tasks below and implement just that.
(1) …  (2) …  (3) …
<assistant> I select (
```
The next-token probabilities of the labels `1 2 3` are read with the goal encoding steered in (`additive_norm`, the
loop's `PROBE_LAYERS` / `PROBE_ALPHA`, at the last `|SEL_STEER_START|` prompt tokens), **averaged over all orderings**
(position bias cancels). Controls on the same prompts: *unsteered* (the prompt's prior) and *base* (the encoding
without the SRC query — kv-only / clean DST). Numeric labels avoid a clash with the code's `a, b, c` variables.

* **(a)** the selected task becomes the single task of a stub probe (as `gaze_select`), then the closed loop.
* **(b)** the probe IS the selection prompt in one fixed ordering, prefilled with the choice and the code stub:
  `I select (k)` + `<code>` + DST's signature + `return` / the indent — so the probe's goal heads can read the list
  *and* the decision.

Both reuse the closed-loop setup above (patched-run prefill, `PF`, `patched_ctx`, the encodings `ENC`) without
changing any of its globals.

In [30]:
# ── knobs ────────────────────────────────────────────────────────────────────
SEL_VEC = "patched"        # encoding that nudges the selection (and is injected if SEL_INJECT): "patched" | "contrast"
SEL_STEER_START = -1       # steer the last |n| tokens of the selection prompt (-1 = only the final " (")
SEL_INJECT = False         # also inject SEL_VEC into the (a)/(b) probes during the loop (as in the stub probe)
SEL_SEED = 0               # (b): which ordering of the options the selection-as-probe prompt uses

import itertools, random

SEL_TASKS = list(KV_ORDER if USE_KV else DST_ORDER)            # the list the goal heads read (incl. the written task)
_LABEL_IDS = [mt.tokenizer(str(i + 1), add_special_tokens=False)["input_ids"][-1] for i in range(len(SEL_TASKS))]
_unit = lambda v: (v.float() / v.float().norm()).float()
SEL_UNITS = {"unsteered": None, "base": _unit(ENC["base"]), SEL_VEC: _unit(ENC[SEL_VEC])}
SEL_INSTR = "Select one of the tasks below and implement just that."


def sel_prompt(order, assistant="I select ("):
    user = SEL_INSTR + "\n" + "\n".join(f"({i + 1}) {by[TASKS[L]].description}" for i, L in enumerate(order))
    return _probe_chat(user, assistant)


@contextmanager
def steer_with(unit, positions):
    # additive_norm with an explicit unit vector (None = no steering); same rule / layers as the loop's goal_steer
    if unit is None:
        yield
        return
    def edit(output, layer):
        h = output[0] if isinstance(output, tuple) else output
        for q in positions:
            v = h[:, q, :]
            h[:, q, :] = v + PROBE_ALPHA * v.float().norm(dim=-1, keepdim=True).to(v.dtype) * unit.to(v.device, v.dtype)
        return output
    with baukit.TraceDict(mt._model, _STEER_NAMES, edit_output=edit):
        yield


@torch.no_grad()
def selection_scores(unit):
    # {task: mean P(label)} over all orderings + {task: #orderings where it wins}
    probs, wins = {L: [] for L in SEL_TASKS}, {L: 0 for L in SEL_TASKS}
    for order in itertools.permutations(SEL_TASKS):
        tok = prepare_input(prompts=sel_prompt(order), tokenizer=mt)
        n = tok["input_ids"].shape[1]
        with steer_with(unit, list(range(n + SEL_STEER_START, n))):
            logits = mt._model(**{k: v for k, v in tok.items() if k in ("input_ids", "attention_mask")}).logits[0, -1]
        pr = torch.softmax(logits[_LABEL_IDS].float(), -1).cpu()
        for i, L in enumerate(order):
            probs[L].append(float(pr[i]))
        wins[order[int(pr.argmax())]] += 1
    return {L: float(np.mean(v)) for L, v in probs.items()}, wins


SEL = {name: selection_scores(u) for name, u in SEL_UNITS.items()}
SELECTED = max(SEL[SEL_VEC][0], key=SEL[SEL_VEC][0].get)
n_perm = len(list(itertools.permutations(SEL_TASKS)))
t = Table(title=f"k-way selection · mean P(label) over {n_perm} orderings (wins in brackets) · expect {EXPECT_PATCHED}",
          highlight=False)
t.add_column("steer"); [t.add_column(TN(L), justify="right") for L in SEL_TASKS]
for name, (pr, w) in SEL.items():
    top = max(pr, key=pr.get)
    t.add_row(name, *[(f"[bold]{pr[L]:.3f}[/bold]" if L == top else f"{pr[L]:.3f}") + f" ({w[L]})" for L in SEL_TASKS])
rich_print(t)
rich_print(f"selected by {SEL_VEC}: [bold]{TN(SELECTED)}[/bold]   (gaze argmax at the patched pivot: "
           f"{TN(max(GAZE['patched'][0], key=GAZE['patched'][0].get))})")


@torch.no_grad()
def closed_loop_with(p_tok, steer_unit, steer_pos, verbose=True, tag=""):
    # the closed loop of the section above with an explicit probe: same patched run (PF / patched_ctx), q_caus from
    # p_tok steered with steer_unit (None = no injection) at steer_pos (+ every appended token if PROBE_STEER="all")
    eos = mt.tokenizer.eos_token_id
    ids, am = dst_tok["input_ids"], dst_tok["attention_mask"]
    with patched_ctx(pf=PF):
        out = mt._model(input_ids=ids, attention_mask=am, use_cache=True)
    pkv_d, nxt = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
    p_am = p_tok["attention_mask"]
    with steer_with(steer_unit, steer_pos):
        pout = mt._model(input_ids=p_tok["input_ids"], attention_mask=p_am, use_cache=True)
    pkv_p, gen, steps = pout.past_key_values, [int(nxt)], []
    while len(gen) < LOOP_MAX and int(nxt) != eos and "\n" not in mt.tokenizer.decode(gen):
        p_am = _ext(p_am)
        with steer_with(steer_unit if PROBE_STEER == "all" else None, [-1]), \
             baukit.TraceDict(mt._model, list(_Q_NAMES.values()), retain_output=True) as tr:
            pout = mt._model(input_ids=nxt, attention_mask=p_am, past_key_values=pkv_p, use_cache=True)
        pkv_p = pout.past_key_values
        q_caus = {l: tr[n].output[0, -1:].detach().float().cpu() for l, n in _Q_NAMES.items()}
        am = _ext(am)
        with patched_ctx(pf=[dict(which="q", cache=q_caus, heads=_qh, positions=[-1])]):
            out = mt._model(input_ids=nxt, attention_mask=am, past_key_values=pkv_d, use_cache=True)
        pkv_d = out.past_key_values
        steps.append(dict(k=len(gen), tok=mt.tokenizer.decode(gen[-1:]), q=q_caus))
        nxt = out.logits[:, -1].argmax(-1, keepdim=True)
        gen.append(int(nxt))
    text = mt.tokenizer.decode(gen, skip_special_tokens=True)
    if verbose:
        rich_print(f"[bold magenta]closed loop {tag} (expect {EXPECT_PATCHED}):[/bold magenta] {escape(text.split(chr(10))[0])}")
    return text, steps, gen


def loop_gaze_table(steps, gen, title):
    # goal-head gaze over the patched task span at the pivot and every loop token (replay with the same patches)
    m_ = len(steps)
    rp = TokenizerOutput(data=dict(
        input_ids=torch.cat([dst_tok["input_ids"], torch.tensor([gen[:m_]], device=dst_tok["input_ids"].device)], 1),
        attention_mask=torch.ones(1, L0 + m_, dtype=dst_tok["attention_mask"].dtype, device=dst_tok["attention_mask"].device)))
    spec = dict(which="q", heads=_qh, positions=[L0 + k for k in range(m_)],
                cache={l: torch.cat([s["q"][l] for s in steps], 0) for l in Q_LAYERS}) if m_ else None
    info = get_attention_matrices(rp, mt, value_weighted=True, patch_interface=ro_iface(PF + ([spec] if spec else []), boost=False))
    info = info[0] if isinstance(info, list) else info
    rows = [("pivot", pivot_attn(info, list(GAZE_HEADS), _span, piv_dst))]
    rows += [(f"t_{s['k']} {s['tok']!r}", pivot_attn(info, list(GAZE_HEADS), _span, L0 + s["k"] - 1)) for s in steps]
    tt = Table(title=title, highlight=False)
    tt.add_column("query"); [tt.add_column(L, justify="right") for L in _items]
    for name, a in rows:
        g_ = _per_item(a); top = max(g_, key=g_.get)
        tt.add_row(name, *[f"[bold]{v:.4f}[/bold]" if L == top else f"{v:.4f}" for L, v in g_.items()])
    rich_print(tt)
    return rows

k-way selection · mean P(label) over 6 orderings (wins in brackets) · expect set D = 
                  'total_vowels' / count Y = 'count_ends_consonant'                  
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ steer     ┃ A = 'longest_word_length' ┃ B = 'distinct_count' ┃ D = 'total_vowels' ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ unsteered │                 0.368 (3) │            0.178 (0) │          0.454 (3) │
│ base      │                 0.290 (2) │            0.191 (0) │          0.519 (4) │
│ patched   │                 0.218 (1) │            0.189 (1) │          0.593 (4) │
└───────────┴───────────────────────────┴──────────────────────┴────────────────────┘

selected by patched: D = 'total_vowels'   (gaze argmax at the patched pivot: D = 'total_vowels')

### (a) Selected task → single-task stub probe → closed loop

In [31]:
# the stub probe whose user turn names only the SELECTED task (as gaze_select, but chosen by the goal encoding)
SEL_A_PROMPT = _probe_chat(f"You are implementing a simple Python function that computes "
                           f"{by[TASKS[SELECTED]].description}. Just write the function body, without any comments or "
                           f"explanations.", PROBE_PREFILL if PROBE_PROMPT != "self" else
                           DATA._signature(DATATYPE, dst_inst.var_name, "python") + ("\n    return" if PIVOT == "var_eq" else "\n    "))

print(SEL_A_PROMPT)
sel_a_tok = prepare_input(prompts=SEL_A_PROMPT, tokenizer=mt)
_na = sel_a_tok["input_ids"].shape[1]
_unit_a = SEL_UNITS[SEL_VEC] if SEL_INJECT else None
rich_print(f"(a) probe task = {TN(SELECTED)} · injection {'on (' + SEL_VEC + ')' if SEL_INJECT else 'OFF'}")
GEN["select_a"], SEL_A_STEPS, SEL_A_IDS = closed_loop_with(
    sel_a_tok, _unit_a, list(range(_na + PROBE_STEER_START, _na)), tag="(a)")
SEL_A_GAZE = loop_gaze_table(SEL_A_STEPS, SEL_A_IDS, f"(a) closed-loop gaze over {PATCHED_LABEL} — argmax in bold")

<|im_start|>system
You are a helpful assistant proficient in programming.<|im_end|>
<|im_start|>user
You are implementing a simple Python function that computes the total number of vowels across all words. Just write the function body, without any comments or explanations.<|im_end|>
<|im_start|>assistant
def func(words: list[str]):
    return


(a) probe task = D = 'total_vowels' · injection OFF

closed loop (a) (expect set D = 'total_vowels' / count Y = 'count_ends_consonant'):  len(set(''.join(words)))

  (a) closed-loop gaze over p_kv tasks —   
              argmax in bold               
┏━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃ query        ┃      A ┃      B ┃      D ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
│ pivot        │ 0.0254 │ 0.0560 │ 0.0939 │
│ t_1 ' len'   │ 0.0410 │ 0.0437 │ 0.0558 │
│ t_2 '(set'   │ 0.0626 │ 0.0575 │ 0.0831 │
│ t_3 "('"     │ 0.0541 │ 0.0607 │ 0.0618 │
│ t_4 "'."     │ 0.0448 │ 0.0470 │ 0.0498 │
│ t_5 'join'   │ 0.0398 │ 0.0391 │ 0.0407 │
│ t_6 '(words' │ 0.0294 │ 0.0397 │ 0.0497 │
└──────────────┴────────┴────────┴────────┘

### (b) Selection prompt as the probe — prefilled with the choice and the code stub

In [32]:
# one fixed ordering (SEL_SEED); the selected task's label in that ordering is prefilled — no free generation
_order_b = list(SEL_TASKS); random.Random(SEL_SEED).shuffle(_order_b)
_k = _order_b.index(SELECTED) + 1
_stub_b = DATA._signature(DATATYPE, dst_inst.var_name, "python") + ("\n    return" if PIVOT == "var_eq" else "\n    ")
SEL_B_PROMPT = sel_prompt(_order_b, assistant=f"I select ({_k})\n\n<code>\n" + _stub_b)
print(SEL_B_PROMPT)

sel_b_tok = prepare_input(prompts=SEL_B_PROMPT, tokenizer=mt)
_nb = sel_b_tok["input_ids"].shape[1]
_unit_b = SEL_UNITS[SEL_VEC] if SEL_INJECT else None
rich_print(f"(b) ordering {[TN(L) for L in _order_b]} · prefilled 'I select ({_k})' = {TN(SELECTED)} · injection "
           f"{'on (' + SEL_VEC + ')' if SEL_INJECT else 'OFF'} · pivot {_tok_at(sel_b_tok, _nb - 1)!r}")
GEN["select_b"], SEL_B_STEPS, SEL_B_IDS = closed_loop_with(
    sel_b_tok, _unit_b, list(range(_nb + PROBE_STEER_START, _nb)), tag="(b)")
SEL_B_GAZE = loop_gaze_table(SEL_B_STEPS, SEL_B_IDS, f"(b) closed-loop gaze over {PATCHED_LABEL} — argmax in bold")

_line = lambda s: s.split("\n")[0]
rich_print(f"[sea_green3]destination:[/sea_green3] {escape(_line(GEN['destination']))}\n"
           f"[salmon1]re-clamp:[/salmon1]    {escape(_line(GEN['patched']))}\n"
           f"[magenta](a) select→stub:[/magenta] {escape(_line(GEN['select_a']))}\n"
           f"[magenta](b) select-as-probe:[/magenta] {escape(_line(GEN['select_b']))}")

<|im_start|>system
You are a helpful assistant proficient in programming.<|im_end|>
<|im_start|>user
Select one of the tasks below and implement just that.
(1) the length of the longest word
(2) the total number of vowels across all words
(3) the number of distinct words<|im_end|>
<|im_start|>assistant
I select (2)

<code>
def func(words: list[str]):
    return


(b) ordering ["A = 'longest_word_length'", "D = 'total_vowels'", "B = 'distinct_count'"] · prefilled 'I select (2)'
= D = 'total_vowels' · injection OFF · pivot ' return'

closed loop (b) (expect set D = 'total_vowels' / count Y = 'count_ends_consonant'):  
len(set(''.join(words).lower()))

  (b) closed-loop gaze over p_kv tasks —   
              argmax in bold               
┏━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃ query        ┃      A ┃      B ┃      D ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
│ pivot        │ 0.0254 │ 0.0556 │ 0.0938 │
│ t_1 ' len'   │ 0.0321 │ 0.0406 │ 0.0485 │
│ t_2 '(set'   │ 0.0379 │ 0.0513 │ 0.0908 │
│ t_3 "('"     │ 0.0354 │ 0.0541 │ 0.0606 │
│ t_4 "'."     │ 0.0298 │ 0.0450 │ 0.0520 │
│ t_5 'join'   │ 0.0239 │ 0.0309 │ 0.0348 │
│ t_6 '(words' │ 0.0238 │ 0.0363 │ 0.0434 │
│ t_7 ').'     │ 0.0255 │ 0.0418 │ 0.0509 │
│ t_8 'lower'  │ 0.0184 │ 0.0255 │ 0.0328 │
└──────────────┴────────┴────────┴────────┘

destination:  sum(len(w) for w in words)
re-clamp:     len(set(words))
(a) select→stub:  len(set(''.join(words)))
(b) select-as-probe:  len(set(''.join(words).lower()))